# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*

## 👥 Team Information

| | |
|---|---|
| **Course** | PROG8245 – Information Retrieval |
| **Workshop** | Inverted Matrix Workshop |
| **Team #** | *Group 3* |

### Team Members
1. *Ali Cihan Ozdemir - 9091405*
2. *Roshan Bartaula did not come to clas and contributed nothing*
3. *N/A*

---


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Manual IR and Inverted Matrix coding + Markdown documentation (work as teams)
3. **Push to GitHub** *(15 min)* – Teams commit and push initial notebooks. **Make sure to include your names so it is easy to identify the team that developed the Min-Max code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(15 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. **Push to the course GitHub** *(automated)* – Find the last code cell in this notebook, update your team number, and run the cell. It will push your notebook to the repo at `submissions/team#`

## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and Inverted Index.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [33]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'sample_docs', max_posts=20)

Saved 20 blog posts to sample_docs


In [34]:
# Example: Load text files from a folder
import os

def load_documents(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

documents = load_documents('sample_docs/')
print(f"Loaded {len(documents)} documents.")


Loaded 20 documents.


### 💬 Team Talking Point – Step 1: Verifying Vocabulary Size

> **Why vocabulary size matters:** The instructor noted that a vocabulary of 2,000+ unique words is required for a robust index. However, an RSS feed returns HTML-wrapped summaries rather than full article bodies. This means the *raw* word count can be inflated by HTML tags and boilerplate text (e.g., `href`, `div`, `span`). A meaningful vocabulary check should be performed **after** tokenization and normalization, not on raw text.
>
> **Our approach:** We count unique vocabulary both before and after normalization so we can observe exactly how much the pipeline reduces the vocabulary through stop-word removal and stemming. This helps us verify that we still meet the 2,000-word threshold after processing, and flags whether additional documents are needed.


In [35]:
# Team Talking Point 1 - Vocabulary Size Verification
import re

def count_raw_vocabulary(documents):
    """Count unique words across all documents BEFORE normalization."""
    all_words = set()
    for doc in documents:
        words = re.findall(r'\b\w+\b', doc.lower())
        all_words.update(words)
    return all_words

raw_vocab = count_raw_vocabulary(documents)
print(f"Raw vocabulary (before normalization): {len(raw_vocab):,} unique words")

threshold = 2000
if len(raw_vocab) >= threshold:
    print(f"Vocabulary threshold of {threshold:,} words MET ({len(raw_vocab):,} unique terms found).")
else:
    shortfall = threshold - len(raw_vocab)
    print(f"Vocabulary threshold NOT met. Need {shortfall:,} more unique words.")
    print("Consider fetching more documents or using a richer RSS source.")

sample = sorted(list(raw_vocab))[:20]
print(f"\nSample vocabulary (first 20 sorted): {sample}")

Raw vocabulary (before normalization): 2,958 unique words
Vocabulary threshold of 2,000 words MET (2,958 unique terms found).

Sample vocabulary (first 20 sorted): ['0', '000', '02', '03', '03uxw6o4clkp7y', '04', '06', '07', '08', '0811', '09', '097d8c', '0b5394', '0kazywlbx252qf_ts', '0pt', '0px', '1', '10', '100', '100kπ']


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [36]:
import re

def tokenize(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])  # Preview first 20 tokens


['announcing', 'python', 'software', 'foundation', 'fellow', 'members', 'for', 'q3', '2025', 'p', 'span', 'face', 'arial', 'sans', 'serif', 'style', 'background', 'color', 'white', 'color']


### 💬 Team Talking Point – Step 2: Tokenizer Edge Cases

> **Limitation of the base tokenizer:** The regex pattern `\b\w+\b` is effective for standard words but has known blind spots:
>
> | Input | `\b\w+\b` produces | Better handling |
> |---|---|---|
> | `don't` | `don`, `t` | Expand contraction first |
> | `state-of-the-art` | `state`, `of`, `the`, `art` | Hyphen-joined phrase is split |
> | `U.S.A.` | `U`, `S`, `A` | Abbreviation is shattered |
> | `2025` | `2025` | Numbers retained (may inflate vocabulary) |
>
> For a production IR system, contractions and hyphenated compounds deserve special handling. The enhanced tokenizer below demonstrates contraction expansion as a practical improvement.

The comparison focuses on two token parsing strategies: fixed-length parsing and dynamic parsing. In the fixed-length approach, the parser reads a set number of characters at a time, which makes the method simple, fast, and easy to implement. It also has predictable performance, which can be useful in systems where efficiency is critical. However, this approach depends heavily on the input having a strict and consistent format, and even a small shift or unexpected character can cause errors in all following tokens. In contrast, dynamic parsing determines token boundaries based on the content itself, allowing it to recognize words, numbers, or symbols more accurately. This makes it more flexible and reliable when handling variable or real-world input. On the downside, dynamic parsing is more complex to design, may require more processing, and can be slightly slower compared to the fixed approach. Overall, fixed-length parsing offers simplicity and speed but lacks flexibility, while dynamic parsing provides adaptability and accuracy at the cost of increased complexity.

In [37]:
# Team Talking Point 2 - Tokenizer Edge Cases
import re

# Demonstrate edge cases with the base tokenizer
edge_cases = [
    "don't stop learning",
    "state-of-the-art machine learning",
    "U.S.A. leads in AI research",
    "Python 3.12 was released in 2023",
]

print("=== Base Tokenizer Edge Cases ===")
for text in edge_cases:
    result = re.findall(r'\b\w+\b', text.lower())
    print(f"  Input : {text}")
    print(f"  Tokens: {result}")
    print()

# Enhanced tokenizer: expand contractions before splitting
CONTRACTIONS = {
    "don't": "do not", "doesn't": "does not", "can't": "cannot",
    "won't": "will not", "it's": "it is", "i'm": "i am",
    "they're": "they are", "we're": "we are", "you're": "you are",
}

def enhanced_tokenize(text):
    """Expand common contractions, then tokenize."""
    text = text.lower()
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    return re.findall(r'\b\w+\b', text)

print("=== Enhanced Tokenizer (contraction expansion) ===")
for text in edge_cases[:2]:
    result = enhanced_tokenize(text)
    print(f"  Input : {text}")
    print(f"  Tokens: {result}")
    print()

print("Conclusion: For this workshop we use the base tokenizer for simplicity,")
print("but a production system should handle contractions and abbreviations explicitly.")

=== Base Tokenizer Edge Cases ===
  Input : don't stop learning
  Tokens: ['don', 't', 'stop', 'learning']

  Input : state-of-the-art machine learning
  Tokens: ['state', 'of', 'the', 'art', 'machine', 'learning']

  Input : U.S.A. leads in AI research
  Tokens: ['u', 's', 'a', 'leads', 'in', 'ai', 'research']

  Input : Python 3.12 was released in 2023
  Tokens: ['python', '3', '12', 'was', 'released', 'in', '2023']

=== Enhanced Tokenizer (contraction expansion) ===
  Input : don't stop learning
  Tokens: ['do', 'not', 'stop', 'learning']

  Input : state-of-the-art machine learning
  Tokens: ['state', 'of', 'the', 'art', 'machine', 'learning']

Conclusion: For this workshop we use the base tokenizer for simplicity,
but a production system should handle contractions and abbreviations explicitly.


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [38]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize_tokens(tokens):
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])


['announc', 'python', 'softwar', 'foundat', 'fellow', 'member', 'q3', '2025', 'p', 'span', 'face', 'arial', 'san', 'serif', 'style', 'background', 'color', 'white', 'color', 'black']


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alicihanozdemir/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 💬 Team Talking Point – Step 3: Stemming vs. Lemmatization

> **Why we chose stemming (Porter Stemmer) — and what we trade away:**
>
> The Porter Stemmer is fast and rule-based, but it applies mechanical suffix-stripping that occasionally produces non-words (e.g., `"university"` → `"univers"`, `"studies"` → `"studi"`). These stems are internally consistent but not human-readable.
>
> **Lemmatization** uses a vocabulary lookup to return the true dictionary base form (e.g., `"studies"` → `"study"`, `"running"` → `"run"`). It is more accurate but slower and requires a corpus (WordNet).
>
> | | Porter Stemmer | WordNet Lemmatizer |
> |---|---|---|
> | Speed | Fast | Slower |
> | Output | May be non-words | Always real words |
> | Best for | IR / search index | NLP / text analysis |
> | `"running"` | `"run"` | `"run"` |
> | `"studies"` | `"studi"` | `"study"` |
>
> For an Inverted Index focused on retrieval speed, stemming is the standard choice. For downstream tasks like sentiment analysis, lemmatization is preferred.


In [39]:
# Team Talking Point 3 - Stemming vs. Lemmatization Comparison
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = [
    "running", "studies", "university", "caring", "better",
    "wolves", "programming", "releases", "founded", "corpora"
]

print(f"{'Word':<15} {'Porter Stem':<18} {'WordNet Lemma':<18}")
print("-" * 52)
for word in test_words:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos='v')
    diff = "<-- differs" if stem != lemma else ""
    print(f"{word:<15} {stem:<18} {lemma:<18} {diff}")

# Verify vocabulary size after normalization (connects back to Talking Point 1)
all_norm_tokens = []
for doc in documents:
    all_norm_tokens.extend(normalize_tokens(tokenize(doc)))

normalized_vocab = set(all_norm_tokens)
print(f"\nVocabulary AFTER normalization : {len(normalized_vocab):,} unique stems")
print(f"Vocabulary BEFORE normalization: {len(raw_vocab):,} raw words")
print(f"Reduction: {len(raw_vocab) - len(normalized_vocab):,} terms removed by stemming & stop-word removal")

Word            Porter Stem        WordNet Lemma     
----------------------------------------------------
running         run                run                
studies         studi              study              <-- differs
university      univers            university         <-- differs
caring          care               care               
better          better             better             
wolves          wolv               wolves             <-- differs
programming     program            program            
releases        releas             release            <-- differs
founded         found              found              
corpora         corpora            corpora            

Vocabulary AFTER normalization : 2,130 unique stems
Vocabulary BEFORE normalization: 2,958 raw words
Reduction: 828 terms removed by stemming & stop-word removal


## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [40]:
from collections import defaultdict

def build_inverted_index(documents):
    index = defaultdict(list)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        seen = set()
        for token in tokens:
            if token not in seen:
                index[token].append(doc_id)
                seen.add(token)
    return dict(index)

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))  # Preview first 10 terms


{'announc': [0, 2, 3, 4, 6, 8, 9, 10, 11, 14, 16, 17, 19], 'python': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'softwar': [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 18, 19], 'foundat': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 18, 19], 'fellow': [0, 2, 8], 'member': [0, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'q3': [0], '2025': [0, 1, 6, 8, 9, 10, 11, 12, 14, 15, 16, 19], 'p': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19], 'span': [0, 4, 7, 8, 9, 10, 19]}


### 💬 Team Talking Point – Step 4: Index Statistics & Efficiency

> **Evaluating index quality:** After building the inverted index, it is important to assess its structure to understand retrieval effectiveness. Two key metrics are:
>
> - **Average postings list length:** the average number of documents each term maps to. A very high average (close to the corpus size) signals that many terms appear almost everywhere, reducing discriminative power.
> - **Singleton terms:** terms appearing in only one document. A large proportion of singletons is expected in small corpora and corresponds to Zipf's Law — most words in natural language are rare.
>
> Terms that appear in *every* document carry no discriminative power and are effectively second-tier stop words — a useful insight for refining the normalization pipeline.


In [41]:
# Team Talking Point 4 - Inverted Index Statistics
import statistics

postings_lengths = [len(v) for v in inverted_index.values()]
total_terms = len(inverted_index)

print("=== Inverted Index Statistics ===")
print(f"  Total unique terms (stems)   : {total_terms:,}")
print(f"  Total documents              : {len(documents)}")
print(f"  Avg postings list length     : {statistics.mean(postings_lengths):.2f} docs/term")
print(f"  Median postings list length  : {statistics.median(postings_lengths):.1f} docs/term")
print(f"  Max postings list length     : {max(postings_lengths)} docs")
print(f"  Min postings list length     : {min(postings_lengths)} doc")

# Terms that appear in ALL documents
ubiquitous = [term for term, docs in inverted_index.items() if len(docs) == len(documents)]
print(f"\n  Terms in ALL {len(documents)} documents: {ubiquitous[:10]}")
if ubiquitous:
    print("  These carry no discriminative power — consider adding to stop-word list.")

# Top-10 most frequent terms
sorted_terms = sorted(inverted_index.items(), key=lambda x: len(x[1]), reverse=True)
print("\n  Top 10 most frequent terms:")
print(f"  {'Term':<15} {'Doc Count':>10}")
print("  " + "-" * 27)
for term, docs in sorted_terms[:10]:
    bar = '#' * len(docs)
    print(f"  {term:<15} {len(docs):>10}  {bar}")

# Singleton terms
singletons = [t for t, d in inverted_index.items() if len(d) == 1]
print(f"\n  Singleton terms (appear in exactly 1 doc): {len(singletons):,}")
print(f"  ({100 * len(singletons) / total_terms:.1f}% of vocabulary — expected for a small corpus)")

=== Inverted Index Statistics ===
  Total unique terms (stems)   : 2,130
  Total documents              : 20
  Avg postings list length     : 2.59 docs/term
  Median postings list length  : 1.0 docs/term
  Max postings list length     : 20 docs
  Min postings list length     : 1 doc

  Terms in ALL 20 documents: ['python', 'psf', 'org']
  These carry no discriminative power — consider adding to stop-word list.

  Top 10 most frequent terms:
  Term             Doc Count
  ---------------------------
  python                  20  ####################
  psf                     20  ####################
  org                     20  ####################
  p                       19  ###################
  commun                  19  ###################
  href                    19  ###################
  http                    19  ###################
  br                      19  ###################
  support                 19  ###################
  work                    19  #############

## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries.
- Demonstrate that they return the correct documents.


In [42]:
# Phrase query implementation using sliding-window sequence match over normalized tokens

query1 = "Python software"
query2 = "bylaws"

def phrase_query(query, documents):
    results = []
    query_tokens = normalize_tokens(tokenize(query))
    for doc_id, doc in enumerate(documents):
        doc_tokens = normalize_tokens(tokenize(doc))
        for i in range(len(doc_tokens) - len(query_tokens) + 1):
            if doc_tokens[i:i+len(query_tokens)] == query_tokens:
                results.append(doc_id)
                break
    return results

results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

Documents containing the phrase 'Python software': [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 18, 19]
Documents containing the phrase 'bylaws': []


### 💬 Team Talking Point – Phrase Queries: Precision vs. Recall Trade-off

> **How our phrase query works:** The `phrase_query` function normalizes both the query and each document before performing a sliding-window sequence match. Because matching is stem-based, `"Python software"` matches the stemmed sequence `['python', 'softwar']` — catching morphological variants.
>
> **Trade-off:** Stem-based matching improves **recall** (finds more relevant documents) but can reduce **precision** (may occasionally return documents where stemming caused unexpected matches).
>
> **Scalability note:** The current implementation rescans every document for every query — O(N × L) complexity where L is document length. A positional index (built in the Additional Challenge) solves this by merging posting lists instead, making phrase queries O(df₁ + df₂) — far more efficient at scale.


In [43]:
# Team Talking Point - Two additional phrase queries demonstrating range of matching
query3 = "open source"
query4 = "board election"

results3 = phrase_query(query3, documents)
results4 = phrase_query(query4, documents)

print(f"Phrase Query 3 - '{query3}': found in {len(results3)} doc(s) -> {results3}")
print(f"  Stems searched: {normalize_tokens(tokenize(query3))}")
print()
print(f"Phrase Query 4 - '{query4}': found in {len(results4)} doc(s) -> {results4}")
print(f"  Stems searched: {normalize_tokens(tokenize(query4))}")

# Show snippet from first matching document for query3
if results3:
    snippet = documents[results3[0]][:300].replace('\n', ' ')
    print(f"\nSnippet from Document {results3[0]} (query3 match):\n  {snippet}...")
else:
    print("\nNo matches found for query3 in this corpus.")

Phrase Query 3 - 'open source': found in 9 doc(s) -> [1, 2, 3, 4, 9, 10, 14, 16, 19]
  Stems searched: ['open', 'sourc']

Phrase Query 4 - 'board election': found in 1 doc(s) -> [17]
  Stems searched: ['board', 'elect']

Snippet from Document 1 (query3 match):
  CPython Core Dev Sprint 2025 at Arm Cambridge: The biggest one yet  <p><i>Guest blog post authored by <a href="https://www.linkedin.com/in/diegor" target="_blank">Diego Russo</a>, Python Core Developer and Principal Software Engineer at <a href="https://www.arm.com/" target="_blank">Arm</a>.&nbsp;</...


## 🧠 Additional Challenge: Implement Positional Indexes

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [44]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

positional_index = build_positional_index(documents)

sample_term = 'python'
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

Positions for term 'python':
  Document 0: [1, 266, 1587, 1685, 1691, 1696, 1699, 1705, 1709, 1720, 1793, 1846, 1852, 2064]
  Document 1: [25, 66, 385, 469, 484, 740, 893, 953, 982, 1012, 1191, 1273, 1344]
  Document 2: [28, 57, 64, 76, 83, 113, 133, 188, 318, 344, 354, 387]
  Document 3: [23, 32, 38, 150, 300, 309]
  Document 4: [5, 28, 42, 53, 65, 201, 293, 350, 519, 529, 538, 542, 555, 561, 570, 577]
  Document 5: [0, 3, 13, 36, 51, 80, 87, 187, 234, 236, 239, 258, 264]
  Document 6: [6, 16, 34, 60, 146, 187, 203, 210, 273, 332, 336, 339, 562, 634, 654, 747, 752, 763, 772, 776, 789, 795, 806, 815]
  Document 7: [1, 27, 56, 217, 230, 347, 419]
  Document 8: [1, 269, 1200, 1298, 1304, 1309, 1312, 1318, 1322, 1333, 1406, 1459, 1465, 1638]
  Document 9: [0, 129, 300, 307, 341, 390, 399, 508, 995, 1526, 1549, 1559, 1759, 1874, 1902, 2132, 4787, 4841, 4970, 5007, 5021, 5095, 5103]
  Document 10: [1, 24, 33, 52, 68, 83, 99, 123, 126, 163, 179, 188, 201, 214, 221, 240, 244, 254, 317, 356, 4

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

### 📝 Completed Table: Term Frequency (TF) and Inverse Document Frequency (IDF weight)

| Term                           | What is it?                                                                 | How is it used?                          | Pros                                   | Cons                                   |
|-------------------------------|----------------------------------------------------------------------------|------------------------------------------|----------------------------------------|----------------------------------------|
| Term Frequency (TF)            | Number of times a term appears in a document                               | Measures term importance in a document   | Simple, fast, good for ranking         | Doesn't account for common terms       |
| Inverse Document Frequency     | Log ratio of total docs to docs containing the term                        | Weights rare terms higher                | Reduces impact of common words         | Can overweight rare, irrelevant terms  |
| (IDF weight)                   | Used with TF to score relevance: TF-IDF = TF * IDF                         | Improves search relevance                | Highlights unique, important terms     | Requires global corpus statistics      |

Let's implement the Inverse Document Frequency (IDF)

In [45]:
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}
    for term, doc_positions in positional_index.items():
        doc_count = len(doc_positions)
        if doc_count > 0:
            # --- IGNORE ---
            # print(f"Computing IDF for {term} '{total_docs}' / {doc_count}")
            idf_scores[term] = math.log(total_docs / doc_count)
        else:
            idf_scores[term] = 0.0
    return idf_scores

idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# NOTE: We applied stemming so the term may not match exactly if it was stemmed
sample_term = 'softwar'
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

IDF for term 'softwar': 0.22314355131420976


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

In [46]:
import pandas as pd

term = 'softwar'

tf_values = []
for doc_id, doc in enumerate(documents):
    tokens = normalize_tokens(tokenize(doc))
    tf = tokens.count(term)
    tf_values.append(tf)
    print(f"TF for term '{term}' in Document {doc_id}: {tf}")

TF_df = pd.DataFrame({'Document': list(range(len(documents))), 'TF': tf_values})
print(TF_df)

TF for term 'softwar' in Document 0: 1
TF for term 'softwar' in Document 1: 3
TF for term 'softwar' in Document 2: 0
TF for term 'softwar' in Document 3: 6
TF for term 'softwar' in Document 4: 3
TF for term 'softwar' in Document 5: 1
TF for term 'softwar' in Document 6: 3
TF for term 'softwar' in Document 7: 2
TF for term 'softwar' in Document 8: 1
TF for term 'softwar' in Document 9: 4
TF for term 'softwar' in Document 10: 3
TF for term 'softwar' in Document 11: 6
TF for term 'softwar' in Document 12: 0
TF for term 'softwar' in Document 13: 0
TF for term 'softwar' in Document 14: 3
TF for term 'softwar' in Document 15: 1
TF for term 'softwar' in Document 16: 2
TF for term 'softwar' in Document 17: 0
TF for term 'softwar' in Document 18: 2
TF for term 'softwar' in Document 19: 4
    Document  TF
0          0   1
1          1   3
2          2   0
3          3   6
4          4   3
5          5   1
6          6   3
7          7   2
8          8   1
9          9   4
10        10   3
11    

#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.

## 🧠 Reflection
I am testing the submission mechanism.
This section is required.


In [47]:
# Combine TF and IDF to compute TF-IDF for all terms across all documents
import pandas as pd
import math

# --- 1. Query Processor ---
def query_processor(query, inverted_index):
    results = []
    for term in query.split():
        if term in inverted_index:
            results.extend(inverted_index[term])
    return list(set(results))  # return a list

# --- 2. TF-IDF Computation ---
def compute_tf_idf(documents, positional_index, idf_scores):
    """
    Returns tf_idf[term][doc_id] = TF-IDF score.
    TF  = count of term occurrences in document (from positional index).
    IDF = log(N / df) pre-computed in idf_scores.
    """
    tf_idf = {}
    for term, doc_positions in positional_index.items():
        tf_idf[term] = {}
        for doc_id in range(len(documents)):
            tf  = len(doc_positions.get(doc_id, []))  # position count = raw TF
            idf = idf_scores.get(term, 0.0)
            tf_idf[term][doc_id] = round(tf * idf, 4)
    return tf_idf

tf_idf_scores = compute_tf_idf(documents, positional_index, idf_scores)

# --- 3. Display TF-IDF table for 'softwar' ---
term = 'softwar'
idf_val = idf_scores.get(term, 0.0)

rows = []
for doc_id in range(len(documents)):
    tf    = len(positional_index[term].get(doc_id, []))
    tfidf = tf_idf_scores[term][doc_id]
    rows.append({'Document': doc_id, 'TF': tf, 'IDF': round(idf_val, 4), 'TF-IDF': tfidf})

tfidf_df = pd.DataFrame(rows)
print(f"TF-IDF scores for term '{term}' (IDF = {idf_val:.4f}):\n")
print(tfidf_df.to_string(index=False))

# --- 4. Top-5 most discriminative terms in Document 0 ---
print("\n-- Top-5 most discriminative terms in Document 0 (by TF-IDF) --")
doc0_scores = {t: tf_idf_scores[t][0] for t in tf_idf_scores if tf_idf_scores[t][0] > 0}
top5 = sorted(doc0_scores.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (t, score) in enumerate(top5, 1):
    print(f"  {rank}. '{t}' -> TF-IDF = {score}")

# --- 5. TF-IDF Ranked Retrieval Demo ---
print("\n-- TF-IDF Ranked Retrieval Demo --")
query = "python foundation"
query_terms = normalize_tokens(tokenize(query))
print(f"Query: '{query}' -> stems: {query_terms}")

doc_scores = {doc_id: 0.0 for doc_id in range(len(documents))}
for qterm in query_terms:
    if qterm in tf_idf_scores:
        for doc_id, score in tf_idf_scores[qterm].items():
            doc_scores[doc_id] += score

ranked = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
print("Top-5 ranked documents:")
for doc_id, score in ranked[:5]:
    print(f"  Doc {doc_id:>2} - score: {score:.4f}")

TF-IDF scores for term 'softwar' (IDF = 0.2231):

 Document  TF    IDF  TF-IDF
        0   1 0.2231  0.2231
        1   3 0.2231  0.6694
        2   0 0.2231  0.0000
        3   6 0.2231  1.3389
        4   3 0.2231  0.6694
        5   1 0.2231  0.2231
        6   3 0.2231  0.6694
        7   2 0.2231  0.4463
        8   1 0.2231  0.2231
        9   4 0.2231  0.8926
       10   3 0.2231  0.6694
       11   6 0.2231  1.3389
       12   0 0.2231  0.0000
       13   0 0.2231  0.0000
       14   3 0.2231  0.6694
       15   1 0.2231  0.2231
       16   2 0.2231  0.4463
       17   0 0.2231  0.0000
       18   2 0.2231  0.4463
       19   4 0.2231  0.8926

-- Top-5 most discriminative terms in Document 0 (by TF-IDF) --
  1. 'font' -> TF-IDF = 178.188
  2. 'normal' -> TF-IDF = 170.3913
  3. 'color' -> TF-IDF = 140.3869
  4. 'white' -> TF-IDF = 89.094
  5. 'variant' -> TF-IDF = 85.1956

-- TF-IDF Ranked Retrieval Demo --
Query: 'python foundation' -> stems: ['python', 'foundat']
Top-5 ranked 

## 🧠 Push your code to the course GitHub repository

Update your team number and then run the code. 

In [48]:
# ⚠️ UPDATE your team number below before running!
!python submit_assignment.py --notebook IR_InvertedMatrix_Workshop.ipynb --student_id "team#"


python: can't open file '/Users/alicihanozdemir/Documents/AllCodingProjects/PROG8245/IR_InvertedMatrix_Workshop-Sonnet/submit_assignment.py': [Errno 2] No such file or directory
